In [14]:
import asyncio
import websockets
import aiohttp
from queue import Queue
import pyaudio
import openai

In [ ]:
base = "http://127.0.0.1:8000/api/v1/tts/feed_input"
stream_base = "http://127.0.0.1:8000/api/v1/tts/feed_stream"
body = {
    "input": "Womp womp"
}

message = """
This is a really long message, with various sentences and etc. Words, words, words.
Here we will write some information to test out the TTS system, and see if it can parse nicely.
Anyways, that will be the end of this all.
"""

In [8]:
def openai_generator(gen_stream):
    """Generator for OpenAI streaming API.
    Yields sentences as they are completed."""
    # SAMPLE
    queue = Queue()
    for chunk in gen_stream.split(" "):
        queue.put(chunk)
    queue.put(None)
    
    payload = ""
    # for chunk in gen_stream:
    while (content := queue.get()) is not None:
        # print(content, end="-")
        ends = ["?", ".", "!"]
        results = [end in content for end in ends]
        content = content + " "
        if any(results):
            payload += content
            # Get index of last found end in content
            last = max([payload.rindex(ends[i]) for i, x in enumerate(results) if x])
            feed, payload = payload[:last+1], payload[last+1:]
            # print("EOL")
            yield feed
        else:
            payload += content

    if payload.strip() != "":
        yield payload

In [ ]:
audio_sequence = list()

async def consume_audio(response, index):
    buffer = b""
    async for message in response.content.iter_any():
        buffer += message
        if buffer.endswith(b"\n"):
            audio_sequence.put(buffer)
            buffer = b""

    # Replace placeholder
    audio_sequence[index] = buffer
    print("Done consuming audio")


async def main():
    tasks = set()

    # for index, chunk in enumerate(openai_generator(message)):   
    # Put placeholder     
    audio_sequence.append("")
    async with aiohttp.ClientSession() as session:
        # First index has param ?stream=true
        async with session.post(base, json={"input": chunk}, params={"stream": str(index == 0)} ) as response:
            if index == 0:
                print(response.status)
                task = asyncio.create_task(consume_audio(response, index))
                print("Done chunk", index)
                tasks.add(task)
    return
    print("Done all")
    # Play audio
    p = pyaudio.PyAudio()
    stream = p.open(format=pyaudio.paInt16, channels=1, rate=22050, output=True)
    for audio in audio_sequence:
        stream.write(audio)
    stream.stop_stream()
    stream.close()
    p.terminate()

                

await main()

In [13]:
audio_sequence = list()

async def consume_audio(response, index):
    buffer = b""
    async for message in response.content.iter_any():
        buffer += message
        if buffer.endswith(b"\n"):
            audio_sequence.put(buffer)
            buffer = b""

    # Replace placeholder
    audio_sequence[index] = buffer
    print("Done consuming audio")

async def main():
    tasks = set()
    for index, chunk in enumerate(openai_generator(message)):   
        # Put placeholder     
        audio_sequence.append("")
        async with aiohttp.ClientSession() as session:
            # First index has param ?stream=true
            async with session.post(base, json={"input": chunk}, params={"stream": str(index == 0)} ) as response:
                if index == 0:
                    print(response.status)
                    task = asyncio.create_task(consume_audio(response, index))
                    print("Done chunk", index)
                    tasks.add(task)
    
    await asyncio.gather(*tasks)
    print("Done all")
    # Play audio
    p = pyaudio.PyAudio()
    stream = p.open(format=pyaudio.paInt16, channels=1, rate=22050, output=True)
    for audio in audio_sequence:
        stream.write(audio)
    stream.stop_stream()
    stream.close()
    p.terminate()

                

await main()

200
Done chunk 0


ClientConnectionError: Connection closed